In [0]:
# ---------------------------------------------------------------------------------------------------------------
# SCRIPT: 4.3_score_anomalia_ceap
# LOCAL: 3_gold/src/4_gastos_ceap/
# OBJETIVO: Calcula o Z-Score a partir da tabela gerada no script 3_gold/4_gastos_ceap/4.2_fato_ceap_despesas 
# ENTREGA: Score de anomalia: z-score por categoria × estado do deputado
# ---------------------------------------------------------------------------------------------------------------

from pyspark.sql.functions import col, avg, stddev, round, when

#Calcular a média e o desvio padrão por Categoria e UF
stats_df = spark.table("gold_fato_ceap_despesas") \
    .groupBy("categoria", "uf") \
    .agg(
        avg("valor_gasto").alias("media_uf_cat"),
        stddev("valor_gasto").alias("desvio_uf_cat")
    )

# Unir com a Fato e calcular o Z-Score com proteção contra divisão por zero
df_anomalia = spark.table("gold_fato_ceap_despesas") \
    .join(stats_df, ["categoria", "uf"], "inner") \
    .withColumn("z_score", 
        when(col("desvio_uf_cat") > 0, 
             (col("valor_gasto") - col("media_uf_cat")) / col("desvio_uf_cat"))
        .otherwise(0)
    ) \
    .withColumn("z_score", round(col("z_score"), 4)) \
    .orderBy(col("z_score").desc())

# Salvar na Gold
df_anomalia.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_anomalias_ceap")

# Exibindo a tabela final com o Z-Score para identificar anomalias (valor > 3 é uma anomalia)
display(df_anomalia.select("nome_deputado", "uf", "categoria", "valor_gasto", "z_score"))